<a href="https://colab.research.google.com/github/RodolfoFerro/cerebroartificial/blob/main/notebooks/MicroLightCNN_Tutorial.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🎤 Reconocimiento de Emociones en la Voz con Redes Neuronales Convolucionales (CNN)

## Guía paso a paso para entrenar el modelo MicroLightCNN para detección de enojo

**Nivel:** Principiante (preparatoria)  
**Tiempo estimado:** 3-4 horas  
**Requisitos:** Python básico, curiosidad por la inteligencia artificial

---

### ¿Qué vamos a hacer?

En este notebook vamos a construir, paso a paso, un sistema de inteligencia artificial capaz de **escuchar una grabación de voz** y determinar si la persona está **enojada** o si está expresando **otra emoción** (felicidad, tristeza, sorpresa, o neutralidad).

### ¿Por qué es importante?

Este tipo de tecnología tiene aplicaciones reales muy valiosas:
- **Detección temprana de violencia doméstica:** Un dispositivo pequeño podría alertar cuando se detectan patrones de agresión verbal.
- **Salud mental:** Monitorear el estado emocional de pacientes.
- **Asistentes virtuales:** Mejorar la interacción humano-computadora.

### ¿Qué tecnologías usaremos?

| Tecnología | ¿Para qué? |
|------------|------------|
| **Python** | Lenguaje de programación |
| **Librosa** | Procesar audio |
| **NumPy/SciPy** | Operaciones matemáticas |
| **TensorFlow/Keras** | Construir la red neuronal |
| **Matplotlib/Seaborn** | Visualizar datos |
| **Scikit-learn** | Evaluar el modelo |

### Flujo general del proyecto

```
Audio (.wav) → Preprocesamiento → Extracción de MFCCs → Entrenamiento CNN → Predicción (Enojo / Otra emoción)
```

> **Basado en:**
> - Fernandez-Morales et al., *"Compact near Real-Time Anger Detection System through Voice Analysis Implemented on ESP32 Microcontroller"*, IEEE Trans. Instrumentation and Measurement, 2025.
> - Fernandez-Morales et al., *"Compact Hybrid Model for Anger and Speaker Recognition through Voice Analysis Implemented on an ESP32 Microcontroller"* (Manuscript, 2026).


---
## 📚 PARTE 1: Fundamentos Teóricos

Antes de escribir código, entendamos los conceptos clave. No te preocupes si no entiendes todo a la primera — iremos paso a paso.


### 1.1 ¿Qué es el sonido?

El sonido es una **vibración** que viaja por el aire. Cuando hablas, tus cuerdas vocales vibran y crean ondas que llegan al oído (o a un micrófono).

Una computadora representa el sonido como una secuencia de números. Cada número indica la "altura" de la onda en un instante de tiempo. A esto se le llama **señal digital de audio**.

**Conceptos clave:**
- **Frecuencia de muestreo (sample rate):** Cuántas veces por segundo medimos la onda. Usaremos **16,000 Hz** (16,000 muestras por segundo), que es suficiente para capturar la voz humana. ¿Por qué? Porque la voz contiene energía entre 0 y 8,000 Hz, y el **Teorema de Nyquist** nos dice que necesitamos muestrear al menos al doble: 2 × 8,000 = 16,000 Hz.
- **Duración:** Cada audio durará **2 segundos**, así que tendremos 16,000 × 2 = **32,000 muestras** por audio.


### 1.2 ¿Qué son los MFCCs? (Mel-Frequency Cepstral Coefficients)

Los **MFCCs** son la forma en que le "explicamos" el sonido a la computadora. Son como una **huella digital del audio** que captura las características más importantes de la voz.

**¿Por qué no usar el audio directamente?**
- El audio crudo tiene 32,000 números por cada 2 segundos — demasiada información.
- Mucha de esa información es redundante o irrelevante.
- Los MFCCs comprimen el audio en una representación compacta y útil.

**Proceso para calcular MFCCs (simplificado):**

```
Audio crudo (32,000 muestras)
    ↓
1. Pre-énfasis: Resalta frecuencias altas (como las consonantes)
    ↓
2. Ventaneo: Divide el audio en pedacitos de 32ms (ventanas de 512 muestras)
    ↓
3. FFT (Transformada Rápida de Fourier): Convierte cada ventana de "tiempo" a "frecuencia"
    ↓
4. Filtros Mel: Aplica filtros que imitan cómo el oído humano percibe frecuencias
    ↓
5. Logaritmo: Comprime el rango dinámico (como el oído)
    ↓
6. DCT (Transformada Discreta del Coseno): Comprime aún más la información
    ↓
Resultado: Matriz de 20 × 124 coeficientes (luego recortamos a 20 × 63)
```

**Analogía:** Imagina que quieres describir una canción a alguien por mensaje de texto. No enviarías la grabación completa, sino algo como: "Empieza suave, luego sube el volumen, tiene muchos graves, el ritmo es rápido..." Los MFCCs hacen algo similar: resumen las características más importantes del audio.


### 1.3 ¿Qué es una Red Neuronal Convolucional (CNN)?

Una **CNN** es un tipo de inteligencia artificial inspirada en cómo funciona el cerebro para reconocer patrones visuales.

**¿Por qué "convolucional"?**
- Usa una operación matemática llamada **convolución**, que es como pasar un "filtro" sobre los datos para detectar patrones.
- Imagina que tienes una lupa que busca patrones específicos: bordes, texturas, formas...

**Componentes principales:**

| Capa | ¿Qué hace? | Analogía |
|------|-------------|----------|
| **Conv2D** | Detecta patrones locales | Una lupa que busca formas específicas |
| **BatchNormalization** | Normaliza las activaciones entre capas | Ajustar el brillo de una foto antes de analizarla |
| **MaxPooling** | Reduce el tamaño, conserva lo importante | Resumir un párrafo en una oración |
| **Flatten** | Convierte la matriz 2D en un vector 1D | Poner las fichas de un rompecabezas en fila |
| **L2 Normalization** | Normaliza el vector a longitud unitaria | Escalar un mapa para que siempre mida lo mismo |
| **Dense** | Toma decisiones combinando toda la información | El cerebro decide: "esto suena a enojo" |
| **Sigmoid** | Da una probabilidad entre 0 y 1 | "Hay un 85% de probabilidad de que sea enojo" |

### 1.4 Arquitectura MicroLightCNN (Nuestro modelo)

Nuestro modelo está inspirado en **LightCNN**, una red originalmente diseñada para reconocimiento facial bajo condiciones ruidosas. LightCNN usa una activación especial llamada **MFM (Max Feature Map)** que selecciona el canal más activo entre pares de feature maps, promoviendo representaciones dispersas.

Sin embargo, MFM **no es compatible con TensorFlow Lite Micro** (el framework para microcontroladores), así que lo reemplazamos por **ReLU**, que es soportado nativamente. De LightCNN conservamos dos principios clave:
1. **Uso exclusivo de kernels 3×3** — filtros pequeños pero eficientes.
2. **Max-pooling agresivo no superpuesto** antes de las capas densas.

El resultado es **MicroLightCNN**, con solo **19,409 parámetros** y una precisión INT8 del **83.2%** en detección de enojo.

```
Entrada: MFCC (20 × 63 × 1)
         ↓
 ── STEM ──────────────────────────────────
    Conv2D (16 filtros, 3×3, sin bias)       ← Detecta patrones básicos
    BatchNormalization                       ← Estabiliza el entrenamiento
    ReLU                                     ← Activación no lineal
         ↓
 ── BLOQUE 1 ──────────────────────────────
    MaxPooling2D (2×2, stride 2)             ← Reduce dimensiones: 10×31×16
         ↓
 ── BLOQUE 2 ──────────────────────────────
    Conv2D (16 filtros, 3×3, sin bias)       ← Patrones de nivel medio
    BatchNormalization + ReLU
    MaxPooling2D (2×2, stride 2)             ← Reduce a: 5×15×16
         ↓
 ── BLOQUE 3 ──────────────────────────────
    Conv2D (16 filtros, 3×3, sin bias)       ← Patrones de alto nivel
    BatchNormalization + ReLU
    MaxPooling2D (2×2, stride 2)             ← Reduce a: 2×7×16
         ↓
 ── EMBEDDING ─────────────────────────────
    Flatten (224 dimensiones)
    L2 Normalization                         ← Vector unitario
         ↓
 ── CLASIFICACIÓN ─────────────────────────
    Dense (64 neuronas, ReLU)                ← Combina características
    Dense (1 neurona, Sigmoid)               ← Probabilidad de enojo
```

**¿Por qué BatchNormalization?** Normaliza las salidas de cada capa convolucional, lo que acelera el entrenamiento y lo hace más estable. Además permite eliminar el bias en las convoluciones (porque BN ya centra los datos), lo que ahorra parámetros.

**¿Por qué L2 Normalization?** Convierte el vector aplanado en un vector de longitud unitaria. Esto hace que el modelo se enfoque en la *dirección* de las características (patrones) y no en su *magnitud* (volumen), mejorando la generalización.


### 1.5 El Dataset: Emotional Speech Database (ESD)

Usaremos el **ESD** (Emotional Speech Database), que contiene:
- **29 horas** de audio grabado en estudio profesional
- **20 hablantes:** 10 en chino y 10 en inglés
- **5 emociones:** Neutral, Feliz (Happy), Enojado (Angry), Triste (Sad) y Sorpresa (Surprise)
- Cada emoción tiene carpetas de `train`, `test` y `evaluation`

**Estructura de carpetas:**
```
ESD/
└── Emotional_Speech_Dataset/
    ├── Chinese/
    │   ├── Angry/  (train / test / evaluation)
    │   ├── Happy/
    │   ├── Neutral/
    │   ├── Sad/
    │   └── Surprise/
    └── English/
        ├── Angry/  (train / test / evaluation)
        ├── Happy/
        ├── Neutral/
        ├── Sad/
        └── Surprise/
```

> **Nota:** El dataset ESD no viene incluido en este notebook. Debes descargarlo por separado.
> Consulta: https://github.com/HLTSingapore/Emotional-Speech-Data

> **Cita obligatoria:** Zhou, K., Sisman, B., Liu, R., & Li, H. (2021). *Seen and unseen emotional style transfer for voice conversion with a new emotional speech dataset.* ICASSP 2021.


---
## 🔧 PARTE 2: Preparación del Entorno


In [ ]:
# ============================================================
# CELDA 1: Instalación de librerías (ejecutar solo una vez)
# ============================================================
# Si estás en VS Code con entorno virtual:
# pip install numpy scipy matplotlib seaborn librosa tensorflow scikit-learn ipykernel pandas
!pip install -q gdown

print("✅ Librerías listas. ¡Vamos a empezar!")


In [ ]:
# ============================================================
# CELDA 2: Importar librerías
# ============================================================
import gdown
import numpy as np
from scipy.fft import dct
import math
import matplotlib.pyplot as plt
import seaborn as sns
import librosa
import librosa.display
import IPython.display as ipd
import tensorflow as tf
from tensorflow.keras.regularizers import l2
from sklearn.metrics import confusion_matrix, precision_recall_fscore_support
from sklearn.metrics import roc_curve, auc, roc_auc_score
import pandas as pd
import os
import warnings
warnings.filterwarnings('ignore')

print(f"NumPy:      {np.__version__}")
print(f"TensorFlow: {tf.__version__}")
print(f"Librosa:    {librosa.__version__}")
print(f"\n✅ Todas las librerías importadas correctamente.")


---
## 📂 PARTE 3: Cargar el Dataset ESD

**⚠️ IMPORTANTE:** Ajusta la variable `BASE_PATH` para que apunte a donde tienes el dataset ESD.


In [ ]:
# ============================================================
# CELDA 2.5: Descargar DB de Drive
# ============================================================
file_id = "1I1qBHj1pe7oFX19tXuhvMQK5c9haOSKv"
gdown.download(
    id=file_id,
    output="ESD.zip",
    quiet=False
)

print("✅ ¡Dataset descargado!")

In [ ]:
# Descomprimir el archivo
!unzip ESD.zip

print("✅ ¡Dataset descomprimido!")

In [ ]:
# ============================================================
# CELDA 3: Configurar rutas del dataset
# ============================================================

# ⚠️ CAMBIA ESTA RUTA para que apunte a tu carpeta ESD
BASE_PATH = "ESD/Emotional_Speech_Dataset"

# Verificar que la ruta existe
if os.path.exists(BASE_PATH):
    print(f"✅ Dataset encontrado en: {BASE_PATH}")
    for item in sorted(os.listdir(BASE_PATH)):
        print(f"   📁 {item}")
else:
    print(f"❌ No se encontró el dataset en: {BASE_PATH}")
    print("   Modifica BASE_PATH con la ruta correcta.")


In [ ]:
# ============================================================
# CELDA 4: Definir parámetros fundamentales
# ============================================================

# --- Parámetros de Audio ---
sampling_rate = 16000   # 16 kHz (Nyquist: 2 × 8000 Hz)
duration = 2            # 2 segundos por audio
num_muestras = sampling_rate * duration  # 32,000 muestras

# --- Clases del problema (clasificación binaria) ---
clases_vec = [0, 1]
emotion_vec = ["Other", "Angry"]

# --- Emociones e idiomas del dataset ---
emotions = ["Angry", "Happy", "Neutral", "Sad", "Surprise"]
languages = ["English", "Chinese"]

print("📊 Parámetros configurados:")
print(f"   Frecuencia de muestreo: {sampling_rate} Hz")
print(f"   Duración por audio:     {duration} s")
print(f"   Muestras por audio:     {num_muestras:,}")
print(f"   Clases:                 {emotion_vec}")


In [ ]:
# ============================================================
# CELDA 5: Construir rutas automáticamente
# ============================================================
paths = {}
for lang in languages:
    paths[lang] = {}
    for emotion in emotions:
        paths[lang][emotion] = {}
        for split in ["train", "test", "evaluation"]:
            paths[lang][emotion][split] = os.path.join(BASE_PATH, lang, emotion, split)

# Verificar rutas clave
print("🔍 Verificando rutas del dataset...")
for lang in languages:
    path_check = paths[lang]["Angry"]["train"]
    exists = "✅" if os.path.exists(path_check) else "❌"
    print(f"   {exists} {lang}/Angry/train")


---
## 🎵 PARTE 4: Cargar y Preprocesar los Audios

Vamos a:
1. Leer cada archivo `.wav`
2. Detectar dónde empieza realmente la voz (saltar silencios)
3. Estandarizar a exactamente 2 segundos
4. Agregar ruido blanco suave para robustez

> **⚠️ Nota macOS:** Si el dataset fue copiado en macOS, puede generar archivos duplicados como `0012_001549 2.wav`. Antes de correr esta celda, ejecuta en terminal:
> ```bash
> find /ruta/a/tu/ESD -name "* 2.wav" -delete
> ```


In [ ]:
# ============================================================
# CELDA 6: Función para cargar y preprocesar audios
# ============================================================

def loadAudio(pathAudio, label, max_files=None):
    """
    Carga archivos .wav, detecta inicio de voz, recorta a 2s, agrega ruido.

    Parámetros:
        pathAudio: ruta a la carpeta con .wav
        label: 1 = Angry, 0 = Other
        max_files: si se especifica, toma como máximo esa cantidad de
                   archivos (elegidos al azar) en vez de todos los .wav
                   de la carpeta — reduce RAM/tiempo cuando no se van a
                   necesitar todos (p.ej. emociones "Other" que luego se
                   submuestrean para balancear con "Angry")
    Retorna:
        (audios numpy array, labels numpy array)
    """
    files = librosa.util.find_files(pathAudio, ext=['wav'])
    files = np.asarray(files)

    if max_files is not None and len(files) > max_files:
        idx = np.random.choice(len(files), max_files, replace=False)
        files = files[idx]

    Audios = []

    for archivo in files:
        try:
            audio, _ = librosa.load(archivo, sr=sampling_rate, mono=True, duration=duration)
        except Exception as e:
            print(f"   ⚠️ Saltando archivo corrupto: {os.path.basename(archivo)}")
            continue

        # Detectar inicio de voz mediante salto de energía
        spectrogram = np.abs(librosa.stft(audio, n_fft=512, hop_length=256))
        spectrogram_db = librosa.amplitude_to_db(spectrogram, ref=np.max)
        max_energy = np.array([np.mean(spectrogram_db[:, i])
                               for i in range(spectrogram_db.shape[1])])

        start_pos = 0
        for i in range(1, spectrogram_db.shape[1]):
            diff_energy = abs(max_energy[i] - max_energy[i-1])
            if diff_energy >= 5:
                col = spectrogram_db.shape[1]
                muesXcol = int(num_muestras / col)
                start_pos = i * muesXcol
                break

        # Recargar con margen y recortar desde inicio de voz
        audio_full, _ = librosa.load(archivo, sr=sampling_rate, mono=True, duration=duration + 2)
        new_audio = audio_full[start_pos : start_pos + num_muestras]
        Audios.append(new_audio)

    # Estandarizar dimensiones + ruido blanco
    # (float32 explícito: librosa ya carga en float32, pero np.zeros/np.random.normal
    #  son float64 por defecto y "upcastean" todo el arreglo si no se fuerza el dtype)
    Audios_equal_dim = []
    for audio in Audios:
        if audio.shape[0] < num_muestras:
            sample = np.append(audio, np.zeros(num_muestras - audio.shape[0], dtype=np.float32))
        else:
            sample = audio[0:num_muestras]

        mean = np.mean(sample)
        sd = np.std(sample)
        white_noise = np.random.normal(loc=mean, scale=abs(sd/50), size=num_muestras).astype(np.float32)
        Audios_equal_dim.append((sample + white_noise).astype(np.float32))

    return np.asarray(Audios_equal_dim, dtype=np.float32), np.full(len(Audios_equal_dim), label)

print("✅ Función loadAudio definida (con protección contra archivos corruptos, muestreo opcional y dtype float32).")


In [ ]:
# ============================================================
# CELDA 7: Cargar TODOS los audios del dataset
# ============================================================
# ⚠️ TARDA VARIOS MINUTOS
# Para las emociones "Other" (todo menos Angry) solo se carga tanto como
# se va a necesitar para el balanceo de la CELDA 8 (con 40% de margen para
# mantener variedad al submuestrear), en vez de decodificar la carpeta
# completa y descartar casi todo después.

MARGEN_OTHER = 1.4
NUM_EMOCIONES_OTHER = len(emotions) - 1  # todas menos "Angry"

print("⏳ Cargando audios del dataset ESD...\n")

audio_data = {}
for lang in languages:
    audio_data[lang] = {}
    for emotion in emotions:
        audio_data[lang][emotion] = {}

    for split in ["train", "test", "evaluation"]:
        # Referencia: cuántos archivos "Angry" hay para este idioma/split
        angry_path = paths[lang]["Angry"][split]
        num_angry_files = len(librosa.util.find_files(angry_path, ext=['wav'])) if os.path.exists(angry_path) else 0
        cap_other = math.ceil(num_angry_files / NUM_EMOCIONES_OTHER * MARGEN_OTHER) if num_angry_files > 0 else None

        for emotion in emotions:
            label = 1 if emotion == "Angry" else 0
            path = paths[lang][emotion][split]
            max_files = None if emotion == "Angry" else cap_other

            if os.path.exists(path):
                audios, labels = loadAudio(path, label, max_files=max_files)
                audio_data[lang][emotion][split] = {'audios': audios, 'labels': labels}
                tope_str = f" (tope: {max_files})" if max_files else ""
                print(f"   ✅ {lang}/{emotion}/{split}: {audios.shape[0]} audios{tope_str}")
            else:
                print(f"   ⚠️ {lang}/{emotion}/{split}: no encontrada")

print("\n🎉 ¡Todos los audios cargados!")


In [ ]:
# ============================================================
# CELDA 8: Balancear y combinar los datos
# ============================================================

datasets = {}
for split in ["train", "test", "evaluation"]:
    angry_a, angry_l, other_a, other_l = [], [], [], []

    for lang in languages:
        for emotion in emotions:
            data = audio_data[lang][emotion][split]
            if emotion == "Angry":
                angry_a.append(data['audios']); angry_l.append(data['labels'])
            else:
                other_a.append(data['audios']); other_l.append(data['labels'])

    angry_audios = np.concatenate(angry_a); angry_labels = np.concatenate(angry_l)
    other_audios = np.concatenate(other_a); other_labels = np.concatenate(other_l)

    # Balancear: tomar tantos "Other" como "Angry"
    num_angry = len(angry_audios)
    if len(other_audios) > num_angry:
        idx = np.random.choice(len(other_audios), num_angry, replace=False)
        other_audios, other_labels = other_audios[idx], other_labels[idx]

    all_audios = np.concatenate([angry_audios, other_audios])
    all_labels = np.concatenate([angry_labels, other_labels])
    shuffle_idx = np.random.permutation(len(all_audios))
    datasets[split] = {'audios': all_audios[shuffle_idx], 'labels': all_labels[shuffle_idx]}

    print(f"📊 {split:>10}: {len(all_audios):>5} audios "
          f"(Angry: {np.sum(all_labels==1)}, Other: {np.sum(all_labels==0)})")

trainAudios, trainLabels = datasets['train']['audios'], datasets['train']['labels']
testAudios, testLabels = datasets['test']['audios'], datasets['test']['labels']
evalAudios, evalLabels = datasets['evaluation']['audios'], datasets['evaluation']['labels']

print(f"\n✅ Entrenamiento: {trainAudios.shape} | Prueba: {testAudios.shape} | Evaluación: {evalAudios.shape}")


---
## 👀 PARTE 5: Explorar los Datos (Visualización)


In [ ]:
# ============================================================
# CELDA 9: Visualizar señales de audio
# ============================================================
fig, axes = plt.subplots(2, 2, figsize=(16, 8))

idx_angry = np.where(trainLabels == 1)[0][0]
idx_other = np.where(trainLabels == 0)[0][0]

for i, (idx, name, color) in enumerate([(idx_angry, "Angry", 'red'),
                                         (idx_other, "Other", 'blue')]):
    audio = trainAudios[idx]
    axes[i][0].plot(audio, color=color, linewidth=0.5)
    axes[i][0].set_title(f"Forma de onda - {name}", fontsize=12, fontweight='bold')
    axes[i][0].set_xlabel("Muestras"); axes[i][0].set_ylabel("Amplitud")
    axes[i][0].grid(True, alpha=0.3)

    spec = np.abs(librosa.stft(audio, n_fft=512, hop_length=256))
    spec_db = librosa.amplitude_to_db(spec, ref=np.max)
    librosa.display.specshow(spec_db, sr=sampling_rate, hop_length=256,
                              x_axis='time', y_axis='hz', ax=axes[i][1], cmap='magma')
    axes[i][1].set_title(f"Espectrograma - {name}", fontsize=12, fontweight='bold')

plt.tight_layout()
plt.show()

print("🔍 El audio 'Angry' suele tener mayor amplitud y más energía en frecuencias altas.")


---
## 🔬 PARTE 6: Extracción de MFCCs

### Parámetros MFCC:

| Parámetro | Valor | Significado |
|-----------|-------|-------------|
| `n_fft` (tamaño ventana) | 512 | 512/16000 = **32 ms** por ventana |
| `hop_length` (salto) | 256 | 256/16000 = **16 ms** entre ventanas |
| `Mel_bands` | 20 | 20 filtros Mel |
| Frames resultantes | 124 | (32000-512)/256 + 1 = 124 |
| **Recorte central** | 63 | Frames 30 a 92 (la voz está en el centro) |


In [ ]:
# ============================================================
# CELDA 10: Parámetros MFCC
# ============================================================
Size_data = num_muestras        # 32,000
Size_win = 512                  # Ventana FFT (32 ms)
Size_off = 256                  # Hop/salto (16 ms)
total_win_num = int(((Size_data - Size_win) / Size_off) + 1)  # 124 frames

frec_min = 20                   # 20 Hz
frec_max = sampling_rate / 2    # 8000 Hz (Nyquist)
Mel_bands = 20                  # 20 bandas Mel

print(f"📐 Ventana: {Size_win} muestras ({Size_win/sampling_rate*1000:.0f} ms)")
print(f"   Hop: {Size_off} muestras ({Size_off/sampling_rate*1000:.0f} ms)")
print(f"   Frames: {total_win_num} → recorte central → 63")
print(f"   Forma MFCC final: ({Mel_bands} × 63)")


In [ ]:
# ============================================================
# CELDA 11: Funciones auxiliares MFCC
# ============================================================
# Implementación manual que replica el cálculo en C++ del ESP32.

def triangular_fun(lower_lim, medium_lim, upper_lim, points_num):
    """Crea un filtro triangular (forma de triángulo)."""
    coefficients = np.zeros(points_num)
    for j in range(points_num):
        if j >= lower_lim and j <= medium_lim:
            coefficients[j] = (j - lower_lim) / (medium_lim - lower_lim)
        elif j > medium_lim and j <= upper_lim:
            coefficients[j] = (upper_lim - j) / (upper_lim - medium_lim)
    return coefficients

def triangular_filter_mat(frec_min, frec_max, Mel_bands):
    """Crea el banco de 20 filtros Mel triangulares."""
    Mel_num_limits = Mel_bands + 2
    frec_min_mel = 2595 * np.log10(1 + frec_min / 700)
    frec_max_mel = 2595 * np.log10(1 + frec_max / 700)
    separation_mel = (frec_max_mel - frec_min_mel) / (Mel_num_limits - 1)

    mel_limits = np.zeros(Mel_num_limits)
    mel_limits[0] = frec_min_mel
    for i in range(1, Mel_num_limits):
        mel_limits[i] = mel_limits[i-1] + separation_mel

    Hz_limits = 700 * (10 ** (mel_limits / 2595) - 1)

    FFT_bins = np.zeros(Mel_num_limits)
    for i in range(Mel_num_limits):
        FFT_bins[i] = math.floor((Size_win // 2 + 1) * Hz_limits[i] / frec_max)
    FFT_bins[-1] = Size_win // 2

    Trian_mat = np.zeros((Mel_bands, Size_win // 2 + 1))
    for i in range(2, Mel_num_limits):
        Trian_mat[i-2, :] = triangular_fun(FFT_bins[i-2], FFT_bins[i-1], FFT_bins[i], Size_win // 2 + 1)
    return Trian_mat

def preemphasis(signal, coeff=0.97):
    """Pre-énfasis: resalta frecuencias altas. y[n] = x[n] - 0.97·x[n-1]"""
    return np.append(signal[0], signal[1:] - coeff * signal[:-1])

def windowing(Hamming_window, preemphasized_signal):
    """Ventaneo con Hamming: divide la señal en segmentos superpuestos."""
    Win_mat = np.zeros((Size_win, total_win_num))
    cont = 0
    for i in range(total_win_num):
        for j in range(Size_win):
            Win_mat[j][i] = Hamming_window[j] * preemphasized_signal[cont]
            cont += 1
        cont = cont - (Size_win - Size_off)
    return Win_mat

def fft_mat(Win_mat):
    """FFT de cada ventana: tiempo → frecuencia."""
    FFT_mat = np.zeros((1 + Size_win // 2, total_win_num))
    for i in range(total_win_num):
        FFT = abs(np.fft.fft(Win_mat[:, i], n=Size_win))
        FFT_mat[:, i] = FFT[0:1 + Size_win // 2]
    return FFT_mat

# Ventana Hamming
Hamming_window = np.array([0.53836 - 0.46164 * np.cos((2 * np.pi * i) / (Size_win - 1))
                           for i in range(Size_win)])

print("✅ Funciones MFCC definidas (réplica del pipeline C++ del ESP32).")


In [ ]:
# ============================================================
# CELDA 12: Función principal de cálculo de MFCCs
# ============================================================

def calculoMFCCs(Audios):
    """
    Pipeline MFCC completo: Pre-énfasis → Ventaneo → FFT → Mel → Log → DCT → Recorte.
    Entrada: (N, 32000)  Salida: (N, 20, 63)
    """
    mfccs = []
    Trian_mat = triangular_filter_mat(frec_min, frec_max, Mel_bands)

    for idx, audio in enumerate(Audios):
        preemphasized_signal = preemphasis(audio)
        Win_mat = windowing(Hamming_window, preemphasized_signal)
        FFT_mat = fft_mat(Win_mat)
        Mel_mat = np.dot(Trian_mat, FFT_mat)
        Mel_mat_log = np.log(Mel_mat + 1e-10)
        mfcc = dct(Mel_mat_log, type=2, axis=0, norm='ortho')[:Mel_bands]

        # Recorte central: 63 frames del centro
        center_start = (mfcc.shape[1] - 63) // 2
        mfccs.append(mfcc[:, center_start:center_start + 63])

        if (idx + 1) % 500 == 0:
            print(f"   Procesados {idx+1}/{len(Audios)} audios...")

    return np.asarray(mfccs)

print("✅ Función calculoMFCCs definida.")


In [ ]:
# ============================================================
# CELDA 13: Calcular MFCCs — ⚠️ TARDA VARIOS MINUTOS
# ============================================================
print("⏳ Calculando MFCCs de entrenamiento...")
trainMFCCs = calculoMFCCs(trainAudios)
print("⏳ Calculando MFCCs de prueba...")
testMFCCs = calculoMFCCs(testAudios)
print("⏳ Calculando MFCCs de evaluación...")
evalMFCCs = calculoMFCCs(evalAudios)

print(f"\n✅ MFCCs calculados:")
print(f"   Train: {trainMFCCs.shape} | Test: {testMFCCs.shape} | Eval: {evalMFCCs.shape}")


In [ ]:
# ============================================================
# CELDA 13.5: Liberar memoria (el audio crudo ya no se necesita)
# ============================================================
# De aquí en adelante solo se usan los MFCCs (mucho más chicos que el
# audio crudo). `trainAudios`/`testAudios`/`evalAudios` apuntan a los
# mismos arrays que `datasets`, así que hay que borrar ambos nombres
# para que el garbage collector libere la memoria de verdad.
import gc

del audio_data, datasets, trainAudios, testAudios, evalAudios
gc.collect()

print("🧹 Memoria de audio crudo liberada — de aquí en adelante solo se usan los MFCCs.")


In [ ]:
# ============================================================
# CELDA 14: Visualizar MFCCs
# ============================================================
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

idx_a = np.where(trainLabels == 1)[0][0]
idx_o = np.where(trainLabels == 0)[0][0]

im1 = axes[0].imshow(trainMFCCs[idx_a], aspect='auto', origin='lower', cmap='RdBu_r')
axes[0].set_title("MFCC - Angry", fontsize=14, fontweight='bold')
axes[0].set_xlabel("Frames"); axes[0].set_ylabel("Coeficientes Mel")
fig.colorbar(im1, ax=axes[0])

im2 = axes[1].imshow(trainMFCCs[idx_o], aspect='auto', origin='lower', cmap='RdBu_r')
axes[1].set_title("MFCC - Other", fontsize=14, fontweight='bold')
axes[1].set_xlabel("Frames"); axes[1].set_ylabel("Coeficientes Mel")
fig.colorbar(im2, ax=axes[1])

plt.tight_layout()
plt.show()
print("🔍 La red neuronal aprenderá a distinguir estos patrones automáticamente.")


---
## 🧠 PARTE 7: Construir y Entrenar MicroLightCNN

¡El momento más emocionante! Vamos a construir la arquitectura MicroLightCNN tal como se describe en el paper.


In [ ]:
# ============================================================
# CELDA 15: Preparar datos para la CNN
# ============================================================
# Agregar dimensión de canal: (N, 20, 63) → (N, 20, 63, 1)

trainMFCCs_cnn = np.expand_dims(trainMFCCs, axis=-1).astype(np.float32)
testMFCCs_cnn = np.expand_dims(testMFCCs, axis=-1).astype(np.float32)
evalMFCCs_cnn = np.expand_dims(evalMFCCs, axis=-1).astype(np.float32)

print(f"📐 Forma de entrada para la CNN: {trainMFCCs_cnn.shape}")
print(f"   Cada MFCC es una 'imagen' de {trainMFCCs_cnn.shape[1]}×{trainMFCCs_cnn.shape[2]} con {trainMFCCs_cnn.shape[3]} canal")


In [ ]:
# ============================================================
# CELDA 16: Definir la arquitectura MicroLightCNN
# ============================================================
# Arquitectura basada en LightCNN adaptada para ESP32-S3.
# Diferencias clave vs MiniVGG16:
#   - BatchNormalization después de cada Conv2D
#   - use_bias=False en Conv2D (BN ya centra los datos)
#   - 16 filtros en el Stem (no 32)
#   - L2 Normalization después de Flatten
#   - Dense(64) en el head (no dos Dense(32) con Dropout)


from keras.optimizers import Adam
from keras.optimizers import SGD
from keras.optimizers import RMSprop
from tensorflow.python.framework.convert_to_constants import convert_variables_to_constants_v2
import pandas as pd
from sklearn.metrics import precision_recall_fscore_support


#Instancia modelo con arquitectura de la red
model = tf.keras.models.Sequential([

  # ---------- LightCNN Inspired Block 1 ----------
  tf.keras.layers.Conv2D(
      32, (3,3),
      padding='same',
      strides=1,
      activation='relu',
      kernel_regularizer=l2(0.57),
      input_shape=(trainMFCCs.shape[1], trainMFCCs.shape[2], 1)
  ),

  tf.keras.layers.MaxPooling2D(3,3),

  # ---------- LightCNN Inspired Block 2 ----------
  tf.keras.layers.Conv2D(
      16, (3,3),
      padding='same',
      strides=1,
      activation='relu'
  ),

  tf.keras.layers.Conv2D(
      16, (3,3),
      padding='same',
      strides=1,
      activation='relu'
  ),

  tf.keras.layers.MaxPooling2D(3,3),

  # ---------- Embedding Layer (similar a LightCNN) ----------
  tf.keras.layers.Flatten(),

  tf.keras.layers.Dense(
      32,
      activation='relu',
      kernel_regularizer=l2(0.57)
  ),

  # ---------- Classification Head ----------
  tf.keras.layers.Dropout(0.36),

  tf.keras.layers.Dense(
      32,
      activation='relu'
  ),

  tf.keras.layers.Dropout(0.36),

  tf.keras.layers.Dense(
      1,
      activation='sigmoid'
  )
])



#Compila el modelo
new_learning_rate = 0.00047
optimizer = Adam(learning_rate=new_learning_rate)
model.compile(optimizer=optimizer, loss='binary_crossentropy', metrics=['accuracy'])


#Imprime un resumen del modelo
model.summary()


# Obtener el input shape del modelo y establecer el batch_size a 1.
# model.input_shape suele ser algo como (None, height, width, channels)
concrete_input_shape = [1] + list(model.input_shape[1:])

# Crear una función tf.function para el forward pass
run_model = tf.function(lambda x: model(x))

# Obtener la función concreta utilizando el input_shape derivado del modelo
concrete_func = run_model.get_concrete_function(tf.TensorSpec(concrete_input_shape, tf.float32))

# Convertir el modelo a un grafo congelado
frozen_func = convert_variables_to_constants_v2(concrete_func)
graph_def = frozen_func.graph.as_graph_def()

# Usar el profiler de TensorFlow para calcular los FLOPS.
with tf.compat.v1.Session() as sess:
    tf.import_graph_def(graph_def, name='')
    options = tf.compat.v1.profiler.ProfileOptionBuilder.float_operation()
    flops_profile = tf.compat.v1.profiler.profile(sess.graph, options=options)

    total_flops = flops_profile.total_float_ops
    print("FLOPS:", total_flops)



In [ ]:
# ============================================================
# CELDA 17: Entrenar MicroLightCNN — ⚠️ TARDA VARIOS MINUTOS
# ============================================================
from keras.callbacks import ModelCheckpoint

os.makedirs('best_model_checkpoint', exist_ok=True)
epochs = 150

model_checkpoint = ModelCheckpoint(
    'best_model_checkpoint/best_model_epoch_{epoch:02d}.h5',
    monitor='val_loss',
    save_best_only=True,
    save_weights_only=False,
    mode='min',
    verbose=1
)

print(f"🏋️ Entrenando MicroLightCNN por {epochs} épocas...")
print(f"   Train: {trainMFCCs_cnn.shape[0]} muestras | Val: {evalMFCCs_cnn.shape[0]} muestras\n")

history = model.fit(
    trainMFCCs_cnn, trainLabels,
    validation_data=(evalMFCCs_cnn, evalLabels),
    epochs=epochs,
    batch_size=32,
    verbose=2,
    callbacks=[model_checkpoint]
)

print("\n🎉 ¡Entrenamiento completado!")


---
## 📈 PARTE 8: Evaluar el Modelo


In [ ]:
# ============================================================
# CELDA 18: Curvas de entrenamiento
# ============================================================
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

axes[0].plot(history.history['loss'], label='Entrenamiento', linewidth=2, color='steelblue')
axes[0].plot(history.history['val_loss'], label='Validación', linewidth=2, color='tomato')
axes[0].set_xlabel('Época', fontsize=14); axes[0].set_ylabel('Pérdida', fontsize=14)
axes[0].set_title('Pérdida (Loss)', fontsize=16, fontweight='bold')
axes[0].legend(fontsize=12); axes[0].grid(True, alpha=0.3)

axes[1].plot(history.history['accuracy'], label='Entrenamiento', linewidth=2, color='steelblue')
axes[1].plot(history.history['val_accuracy'], label='Validación', linewidth=2, color='tomato')
axes[1].set_xlabel('Época', fontsize=14); axes[1].set_ylabel('Precisión', fontsize=14)
axes[1].set_title('Precisión (Accuracy)', fontsize=16, fontweight='bold')
axes[1].legend(fontsize=12); axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("📖 ¿Cómo leer las gráficas?")
print("   ✅ Líneas cercanas = buen ajuste")
print("   ⚠️ Train mejora pero Val no = sobreajuste")


In [ ]:
# ============================================================
# CELDA 19: Cargar mejor modelo y evaluar
# ============================================================
from keras.models import load_model

checkpoint_dir = 'best_model_checkpoint'
checkpoints = sorted([f for f in os.listdir(checkpoint_dir) if f.endswith('.h5')])

if checkpoints:
    best_model_path = os.path.join(checkpoint_dir, checkpoints[-1])
    print(f"📂 Cargando: {checkpoints[-1]}")
    model = load_model(best_model_path)
else:
    print("⚠️ Usando modelo actual (no se encontraron checkpoints).")

for name, data, labels in [("Entrenamiento", trainMFCCs_cnn, trainLabels),
                             ("Prueba", testMFCCs_cnn, testLabels),
                             ("Evaluación", evalMFCCs_cnn, evalLabels)]:
    loss_val, acc_val = model.evaluate(data, labels, verbose=0)
    print(f"   {name:>15}: Loss={loss_val:.4f}  Accuracy={acc_val*100:.1f}%")


In [ ]:
# ============================================================
# CELDA 20: Matriz de Confusión
# ============================================================
umbral = 0.5
predictions = model.predict(evalMFCCs_cnn)
predictions_bin = (predictions > umbral).astype(int).flatten()

con_mat = confusion_matrix(evalLabels, predictions_bin, labels=clases_vec)
con_mat_df = pd.DataFrame(con_mat, index=emotion_vec, columns=emotion_vec)

plt.figure(figsize=(8, 6))
sns.heatmap(con_mat_df, annot=True, fmt='d', cmap='Blues', linewidths=0.5, annot_kws={'size': 16})
plt.xlabel('Predicción', fontsize=14, fontweight='bold')
plt.ylabel('Valor Real', fontsize=14, fontweight='bold')
plt.title('Matriz de Confusión — MicroLightCNN', fontsize=16, fontweight='bold')
plt.tight_layout(); plt.show()

tn, fp, fn, tp = con_mat.ravel()
print(f"   ✅ Verdaderos Negativos (Other→Other): {tn}")
print(f"   ✅ Verdaderos Positivos (Angry→Angry): {tp}")
print(f"   ❌ Falsos Positivos (Other→Angry):     {fp}")
print(f"   ❌ Falsos Negativos (Angry→Other):     {fn}")


In [ ]:
# ============================================================
# CELDA 21: Curva ROC y métricas detalladas
# ============================================================
fpr, tpr, umbrales_roc = roc_curve(evalLabels, predictions)
area_bajo_curva = roc_auc_score(evalLabels, predictions)

plt.figure(figsize=(8, 6))
plt.plot(fpr, tpr, color='steelblue', lw=2.5, label=f'ROC (AUC = {area_bajo_curva:.2f})')
plt.plot([0, 1], [0, 1], color='gray', linestyle='--', lw=1)
plt.fill_between(fpr, tpr, alpha=0.1, color='steelblue')
plt.xlim([0.0, 1.0]); plt.ylim([0.0, 1.05])
plt.xlabel('Tasa de Falsos Positivos (FPR)', fontsize=14)
plt.ylabel('Tasa de Verdaderos Positivos (TPR)', fontsize=14)
plt.title('Curva ROC — MicroLightCNN', fontsize=16, fontweight='bold')
plt.legend(fontsize=13, loc='lower right'); plt.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

precision, recall, f1, _ = precision_recall_fscore_support(evalLabels, predictions_bin, labels=clases_vec)
metrics_df = pd.DataFrame({
    'Clase': emotion_vec,
    'Precisión': [f"{p:.3f}" for p in precision],
    'Recall': [f"{r:.3f}" for r in recall],
    'F1-Score': [f"{f:.3f}" for f in f1]
})
print("\n📊 Métricas por clase:")
print(metrics_df.to_string(index=False))
print(f"\n   AUC: {area_bajo_curva:.3f} — {'Excelente' if area_bajo_curva > 0.85 else 'Bueno' if area_bajo_curva > 0.75 else 'Necesita mejora'}")


---
## 💾 PARTE 9: Exportar para ESP32

Convertimos el modelo a TensorFlow Lite y luego cuantizamos a INT8 para que quepa en el ESP32-S3.


In [ ]:
# ============================================================
# CELDA 22: Guardar modelo Keras
# ============================================================
import pathlib
os.makedirs('saved_models', exist_ok=True)
model.save('saved_models/MicroLightCNN_Anger_Detection.keras')
print("✅ Modelo Keras guardado.")


In [ ]:
# ============================================================
# CELDA 23: Convertir a TFLite float32
# ============================================================
converter = tf.lite.TFLiteConverter.from_keras_model(model)
tflite_model_float = converter.convert()

tflite_path_float = 'saved_models/MicroLightCNN_float32.tflite'
with open(tflite_path_float, 'wb') as f:
    f.write(tflite_model_float)

size_float = os.path.getsize(tflite_path_float)
print(f"📦 TFLite float32: {size_float:,} bytes ({size_float/1024:.1f} KB)")


In [ ]:
# ============================================================
# CELDA 24: Cuantización a INT8 (para ESP32-S3)
# ============================================================
converter_int8 = tf.lite.TFLiteConverter.from_keras_model(model)
converter_int8.optimizations = [tf.lite.Optimize.DEFAULT]

def representative_data_gen():
    for i in range(min(100, len(trainMFCCs_cnn))):
        yield [np.expand_dims(trainMFCCs_cnn[i], axis=0).astype(np.float32)]

converter_int8.representative_dataset = representative_data_gen
converter_int8.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
converter_int8.inference_input_type = tf.int8
converter_int8.inference_output_type = tf.int8

tflite_model_int8 = converter_int8.convert()

tflite_path_int8 = 'saved_models/MicroLightCNN_int8.tflite'
with open(tflite_path_int8, 'wb') as f:
    f.write(tflite_model_int8)

size_int8 = os.path.getsize(tflite_path_int8)
print(f"📦 TFLite INT8: {size_int8:,} bytes ({size_int8/1024:.1f} KB)")
print(f"   Reducción vs float32: {(1 - size_int8/size_float)*100:.1f}%")
print(f"   Arena estimado ~60 KB < 4,000 KB del ESP32-S3 ✅")


In [ ]:
# ============================================================
# CELDA 25: Convertir a .cc (C++ para firmware ESP32)
# ============================================================
with open(tflite_path_int8, "rb") as f:
    data = f.read()

cc_path = 'model_data.cc'
with open(cc_path, "w") as f:
    f.write('// MicroLightCNN INT8 para ESP32-S3\n')
    f.write('// Generado desde el notebook de entrenamiento\n\n')
    f.write('#include "model_data.h"\n\n')               # ← ESTA LÍNEA FALTABA
    f.write('alignas(16) const unsigned char g_model[] = {\n')

    for i, b in enumerate(data):
        if i % 12 == 0:
            f.write("\n  ")
        f.write(f"0x{b:02x}, ")

    f.write("\n};\n")
    f.write(f"const int g_model_len = {len(data)};\n")

print(f"✅ Archivo C++ generado: {cc_path} ({len(data):,} bytes)")


---
## 🧪 PARTE 10: Verificar Modelo TFLite


In [ ]:
# ============================================================
# CELDA 26: Verificar TFLite
# ============================================================
interpreter = tf.lite.Interpreter(model_path=tflite_path_float)
interpreter.allocate_tensors()
input_index = interpreter.get_input_details()[0]["index"]
output_index = interpreter.get_output_details()[0]["index"]

predictions_tflite = []
for mfcc in evalMFCCs_cnn:
    interpreter.set_tensor(input_index, np.expand_dims(mfcc, axis=0).astype(np.float32))
    interpreter.invoke()
    predictions_tflite.append(interpreter.get_tensor(output_index)[0][0])

accuracy_tflite = np.mean((np.array(predictions_tflite) > 0.5).astype(int) == evalLabels)
print(f"📊 Precisión TFLite: {accuracy_tflite*100:.1f}% (debe ser similar al modelo original)")
